In [9]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np

from build_utils import *

In [10]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"

In [11]:
encoder = TeamEncoder.load(team_encoder_path)

In [12]:
seasons=sorted(ALL_SEASONS)

In [13]:
test_df=pd.DataFrame({
    'home': ["Nott'ham Forest", "Manchester Utd"],
    'away': ['Chelsea', 'Aston Villa'],
    'date': ['2025-05-25', '2025-05-25']
    })

In [14]:
test_dict={'premier_league': test_df}

In [15]:
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)

In [ ]:
def build_all_features_for_competition(competition, test_df, seasons, MODELS_PATH, PROCESSED_DATA_PATH):
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    return all_features

# Example usage:
# all_features = build_all_features_for_competition('premier_league', test_dict['premier_league'], seasons, MODELS_PATH, PROCESSED_DATA_PATH)


In [16]:
all_features

,home,away,date,encoded_home_Arsenal,encoded_home_Bournemouth,encoded_home_Brighton,encoded_home_Burnley,encoded_home_Chelsea,encoded_home_Crystal Palace,encoded_home_Everton,...,away_lag_5_home_goals,away_lag_5_away_goals,away_lag_5_home_corners,away_lag_5_away_corners,away_lag_5_home_cards,away_lag_5_away_cards,away_lag_5_home_shots,away_lag_5_away_shots,away_lag_5_home_sots,away_lag_5_away_sots
0,Manchester Utd,Aston Villa,2025-05-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4,1,7,7,1,3,23,10,9,3
1,Nott'ham Forest,Chelsea,2025-05-25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,2,5,4,1,3,6,13,1,7


In [9]:
already_processed = pd.read_csv(f"/Users/tianqihuang/Documents/GitHub/betbot/data/features/premier_league/all_combined_features_2017-24.csv")

In [10]:
already_processed.loc[(already_processed['home'] == 'Manchester Utd') & (already_processed['away'] == 'Aston Villa'), ['home', 'away', 'date', 'away_lag_5_home_goals', 'away_lag_5_away_goals']]

,home,away,date,away_lag_5_home_goals,away_lag_5_away_goals
421,Manchester Utd,Aston Villa,2019-12-01,NaN,NaN
908,Manchester Utd,Aston Villa,2021-01-01,0.0,0.0
1515,Manchester Utd,Aston Villa,2021-09-25,3.0,1.0
1565,Manchester Utd,Aston Villa,2023-04-30,NaN,NaN
2011,Manchester Utd,Aston Villa,2023-12-26,5.0,1.0
2548,Manchester Utd,Aston Villa,2025-05-25,1.0,1.0
